# 02 - Frequentist A/B Analysis & Multiple Testing
**Milestone 3**

We simulate experiment data, run the correct tests (Welch t-test, two-proportion
z-test, chi-square) and correct for multiple comparisons with Bonferroni and
Benjamini-Hochberg. We also analyse the real **Cookie Cats** mobile-game A/B test.


In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def _find_project_root(start):
    d = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(d, "src")) and os.path.exists(
            os.path.join(d, "src", "config.py")
        ):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return os.path.abspath(start)
        d = parent

PROJECT_ROOT = _find_project_root(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import artifact_path, raw_path, ensure_directories
ensure_directories()
FIGS = artifact_path("figures")
os.makedirs(FIGS, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("FIGURES    :", FIGS)


## 1. Continuous metric -> Welch's t-test

In [ ]:
from src.data_pipeline.simulate import simulate_continuous_ab
from src.core import frequentist as freq
from src.core.conclude import frequentist_conclusion

df_cont = simulate_continuous_ab(n_a=2000, n_b=2000, mean_a=10, relative_lift=0.05, std=5, seed=1)
ctrl = df_cont.loc[df_cont.variant == "A", "revenue"].values
trt = df_cont.loc[df_cont.variant == "B", "revenue"].values

res_t = freq.two_sample_ttest(ctrl, trt)
res_t["p_value"], res_t["absolute_lift"], res_t["cohens_d"]

In [ ]:
frequentist_conclusion(res_t)

## 2. Binary metric -> two-proportion z-test

In [ ]:
from src.data_pipeline.simulate import simulate_binary_ab

df_bin = simulate_binary_ab(n_a=5000, n_b=5000, p_a=0.10, relative_lift=0.20, seed=2)
c = df_bin.loc[df_bin.variant == "A", "converted"]
t = df_bin.loc[df_bin.variant == "B", "converted"]

res_z = freq.two_proportion_ztest(int(c.sum()), len(c), int(t.sum()), len(t))
res_z["rate_control"], res_z["rate_treatment"], res_z["p_value"], res_z["cohens_h"]

## 3. Chi-square test of independence
Equivalent to the z-test for a 2x2 table, but generalises to more categories.

In [ ]:
table = [[int(c.sum()), int(len(c) - c.sum())],
         [int(t.sum()), int(len(t) - t.sum())]]
print("observed table (rows: control/treatment, cols: converted/not):")
print(table)
chi = freq.chi_square_test(table)
{ "chi2": chi["chi2_stat"], "p_value": chi["p_value"], "cramers_v": chi["cramers_v"] }

## 4. Multiple testing across many metrics
Only `metric_0` truly moves; the rest are null. Watch how Bonferroni controls the family-wise error rate and BH controls the false-discovery rate.

In [ ]:
from src.data_pipeline.simulate import simulate_multimetric_ab
from src.core import multiple_testing as mt

df_multi = simulate_multimetric_ab(n_per_arm=3000, seed=7)
metric_cols = [c for c in df_multi.columns if c.startswith("metric_")]

pvals = []
for col in metric_cols:
    cc = df_multi.loc[df_multi.variant == "A", col]
    tt = df_multi.loc[df_multi.variant == "B", col]
    pvals.append(freq.two_proportion_ztest(int(cc.sum()), len(cc), int(tt.sum()), len(tt))["p_value"])

batch = mt.analyze_metric_batch(pvals, alpha=0.05, metric_names=metric_cols)
rows = []
for m, r in batch["by_method"].items():
    rows.append({"method": m, "reject": sum(r["reject"]), "adjusted": [round(x, 4) for x in r["adjusted_pvalues"]]})
pd.DataFrame(rows)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(metric_cols)); w = 0.25
ax.bar(x - w, batch["pvalues"], w, label="raw p", color="#94a3b8")
ax.bar(x, batch["by_method"]["bonferroni"]["adjusted_pvalues"], w, label="bonferroni", color="#4f46e5")
ax.bar(x + w, batch["by_method"]["benjamini-hochberg"]["adjusted_pvalues"], w, label="benjamini-hochberg", color="#0ea5e9")
ax.axhline(0.05, ls="--", color="#dc2626", label="alpha = 0.05")
ax.set_xticks(x); ax.set_xticklabels(metric_cols, rotation=30)
ax.set_ylabel("p-value"); ax.set_title("Raw vs. adjusted p-values"); ax.legend()
fig.tight_layout(); fig.savefig(os.path.join(FIGS, "02_multiple_testing.png"), dpi=120, bbox_inches="tight"); plt.show()

## 5. Real data: Cookie Cats (Kaggle)
The loader downloads via `kagglehub`, caches to `data/raw`, and falls back to a structurally identical simulation when offline.

In [ ]:
from src.data_pipeline import datasets

cats = datasets.load_cookie_cats()
print(cats.shape)
cats.head()

In [ ]:
# gate_30 is the control, gate_40 the treatment
g30 = cats.loc[cats.version == "gate_30"]
g40 = cats.loc[cats.version == "gate_40"]
for metric in ("retention_1", "retention_7"):
    r = freq.two_proportion_ztest(int(g30[metric].sum()), len(g30), int(g40[metric].sum()), len(g40))
    print(f"{metric}: gate_30={r['rate_control']:.4f}  gate_40={r['rate_treatment']:.4f}  "
          f"p={r['p_value']:.4f}  significant={r['significant']}")

## Takeaways
* Match the test to the metric: t-test for continuous, z-test for proportions, chi-square for contingency tables.
* Peeking at many metrics inflates false positives - always correct with Bonferroni (strict) or BH (FDR).
* Cookie Cats retention differences are tiny and generally **not** significant, a classic reminder that "no significant result" is a valid outcome.